In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

METADATA_CSV = "/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/clean_candidate_metadata.csv"

df = pd.read_csv(METADATA_CSV)

# Make sure only files that actually exist are included
df = df[
    df["path"].apply(
        lambda p: isinstance(p, str) and os.path.exists(p)
    )
].copy()

print("Total usable images:", len(df))
print("Total classes:", df["landmark"].nunique())

Total usable images: 5275
Total classes: 80


In [ ]:
class_counts = (
    df.groupby(["district", "landmark"])
      .size()
      .reset_index(name="image_count")
)

print("Total classes:", len(class_counts))

print("\nClasses with <30 images:",
      (class_counts["image_count"] < 30).sum())

print("Classes with 30–40 images:",
      (
          (class_counts["image_count"] >= 30) &
          (class_counts["image_count"] <= 40)
      ).sum())

print("Classes with >40 images:",
      (class_counts["image_count"] > 40).sum())

Total classes: 80

Classes with <30 images: 22
Classes with 30–40 images: 9
Classes with >40 images: 49


In [ ]:
valid_classes = class_counts[
    class_counts["image_count"] >= 30
][["district", "landmark"]]

df_valid = df.merge(
    valid_classes,
    on=["district", "landmark"],
    how="inner"
)

print("Classes retained:", df_valid["landmark"].nunique())
print("Images before balancing:", len(df_valid))

Classes retained: 58
Images before balancing: 5011


In [ ]:
import random

random.seed(42)

balanced_parts = []

for (district, landmark), group in df_valid.groupby(
    ["district", "landmark"]
):

    n_available = len(group)

    # If class already has 30–40 images, keep all
    if 30 <= n_available <= 40:
        n_keep = n_available

    # If class has more than 40, randomly choose 30–40
    else:
        n_keep = random.randint(35, 40)

    selected = group.sample(
        n=n_keep,
        random_state=42
    )

    balanced_parts.append(selected)

balanced_df = pd.concat(
    balanced_parts,
    ignore_index=True
)

print("Final classes:", balanced_df["landmark"].nunique())
print("Final images:", len(balanced_df))

Final classes: 58
Final images: 2137


In [ ]:
final_counts = (
    balanced_df
    .groupby(["district", "landmark"])
    .size()
    .reset_index(name="image_count")
    .sort_values(["district", "landmark"])
)

display(final_counts)

,district,landmark,image_count
0,Bandarban,Boga_Lake,40
1,Bandarban,Chimbuk_Hill,35
2,Bandarban,Golden_Temple_Bandarban,35
3,Bandarban,Keokradong,39
4,Bandarban,Nilachal,40
5,Bandarban,Nilgiri,37
6,Brahmanbaria,Anderson_Canal,37
7,Brahmanbaria,Arifail_Mosque,31
8,Brahmanbaria,Haripur_Barendra_Bari,36
9,Brahmanbaria,Titas_River,36


In [ ]:
print("========================================")
print("FINAL DATASET CHECK")
print("========================================")

print("Total classes:", len(final_counts))
print("Total images:", len(balanced_df))

print(
    "Minimum images/class:",
    final_counts["image_count"].min()
)

print(
    "Maximum images/class:",
    final_counts["image_count"].max()
)

print(
    "Classes below 30:",
    (final_counts["image_count"] < 30).sum()
)

print(
    "Classes above 40:",
    (final_counts["image_count"] > 40).sum()
)

if (
    final_counts["image_count"].min() >= 30
    and
    final_counts["image_count"].max() <= 40
):
    print("\n✓ BALANCING SUCCESSFUL")
    print("✓ Every class contains 30–40 images.")
else:
    print("\n❌ Balancing requirement not satisfied.")

FINAL DATASET CHECK
Total classes: 58
Total images: 2137
Minimum images/class: 31
Maximum images/class: 40
Classes below 30: 0
Classes above 40: 0

✓ BALANCING SUCCESSFUL
✓ Every class contains 30–40 images.


In [ ]:
FINAL_DIR = "/content/drive/MyDrive/Chattogram_Landmark_Dataset/05_final_dataset"

os.makedirs(FINAL_DIR, exist_ok=True)

BALANCED_CSV = os.path.join(
    FINAL_DIR,
    "balanced_metadata.csv"
)

balanced_df.to_csv(
    BALANCED_CSV,
    index=False
)

print("Saved:")
print(BALANCED_CSV)

Saved:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/05_final_dataset/balanced_metadata.csv


In [ ]:
import shutil
import os

IMAGE_ROOT = os.path.join(FINAL_DIR, "images")

os.makedirs(IMAGE_ROOT, exist_ok=True)

copied = 0
failed = 0

for _, row in balanced_df.iterrows():

    source = row["path"]

    destination_dir = os.path.join(
        IMAGE_ROOT,
        str(row["district"]),
        str(row["landmark"])
    )

    os.makedirs(destination_dir, exist_ok=True)

    destination = os.path.join(
        destination_dir,
        os.path.basename(source)
    )

    try:
        shutil.copy2(source, destination)
        copied += 1

    except Exception as e:
        failed += 1
        print("Failed:", source)
        print("Error:", e)

print("========================================")
print("COPY COMPLETE")
print("========================================")
print("Images copied:", copied)
print("Failed:", failed)

COPY COMPLETE
Images copied: 2137
Failed: 0


In [ ]:
import os

folder_counts = []

for district in os.listdir(IMAGE_ROOT):

    district_path = os.path.join(IMAGE_ROOT, district)

    if not os.path.isdir(district_path):
        continue

    for landmark in os.listdir(district_path):

        landmark_path = os.path.join(
            district_path,
            landmark
        )

        if not os.path.isdir(landmark_path):
            continue

        count = len([
            f for f in os.listdir(landmark_path)
            if os.path.isfile(
                os.path.join(landmark_path, f)
            )
        ])

        folder_counts.append({
            "district": district,
            "landmark": landmark,
            "image_count": count
        })

final_folder_df = pd.DataFrame(folder_counts)

display(
    final_folder_df.sort_values(
        ["district", "landmark"]
    )
)

,district,landmark,image_count
0,Bandarban,Boga_Lake,40
1,Bandarban,Chimbuk_Hill,35
2,Bandarban,Golden_Temple_Bandarban,35
3,Bandarban,Keokradong,39
4,Bandarban,Nilachal,40
5,Bandarban,Nilgiri,37
6,Brahmanbaria,Anderson_Canal,37
7,Brahmanbaria,Arifail_Mosque,31
8,Brahmanbaria,Haripur_Barendra_Bari,36
9,Brahmanbaria,Titas_River,36


In [ ]:
print("Minimum:", final_folder_df["image_count"].min())
print("Maximum:", final_folder_df["image_count"].max())

assert final_folder_df["image_count"].min() >= 30
assert final_folder_df["image_count"].max() <= 40

print("\n✓ FINAL DATASET VERIFIED")
print("✓ Every class has between 30 and 40 images.")

Minimum: 31
Maximum: 40

✓ FINAL DATASET VERIFIED
✓ Every class has between 30 and 40 images.
